# RoofTop Regression Scoping

## Purpose

Scope the APTs (Address Points) that were **relocated for ESP** so they can be validated
and, once relocation is confirmed, fed into the Data Correctness (relocation) process.

### Background (SEACO-6171)

In Q1–Q2 2026, a total of **8,569,598 (~8.5M)** APTs were ingested across multiple sources
(e.g. `CATASTRO_APT`), with locations updated according to each respective source. Some of
these ingestions moved existing APTs (originally from Genesis, Nomecalles Opensource,
APT Eustat, esp event_lost) away from their rooftop, causing a regression in the ESP RAC
(Rooftop Accuracy) metrics.

### Scope of this notebook

As the next step in the investigation, this notebook:

1. Determines the exact count of APTs that moved **outside the Building Footprint (BFP)**
   in Q1 and Q2 for ESP.
2. Scopes those relocated APTs.
3. Prepares the scoped set for validation, after which the Data Correctness process for
   relocation can be triggered.

Reference: [SEACO-6171](https://tomtom.atlassian.net/browse/SEACO-6171?focusedCommentId=12144061)

## Approach

The relocated-APT scoping is built step by step from two snapshots (OLD vs LATEST) and the
Building Footprint (BFP) layer:

1. **Load the OLD snapshot** into a Delta table (pin the older revision to compare against).
2. **Load the LATEST snapshot** into a Delta table (resolve the newest revision).
3. **Read the OLD snapshot** Delta table as a `Dataset[OrbisElement]`.
4. **Read the LATEST snapshot** Delta table as a `Dataset[OrbisElement]`.
5. **Filter both snapshots by `countryISO` = `ESP`** (value of the `metadata:country` tag).
6. **Filter the OLD snapshot by `metadata:updated`** — keep only APTs whose `metadata:updated`
   value (an epoch timestamp) is **greater than 1st Feb** (the regression window start).
7. **Left join** the OLD snapshot dataset with the LATEST snapshot dataset on the APT id
   (keep all OLD rows; LATEST columns are null where the APT is missing in the latest).
8. **Add geometry / distance columns:**
   - `old_wkt` and `latest_wkt` — `POINT(lng lat)` built from each snapshot's lat/lng.
   - `distance_in_meters` — great-circle (Haversine) distance between the OLD and LATEST point.
   - `hookline` — `LINESTRING(old -> latest)` connecting the two points.
9. **Load the Building Footprint (BFP) snapshot.**
10. **Spatially check the OLD location** of each APT against the BFP — was the OLD APT
    point inside a building footprint?
11. **Final dataset** contains, per APT:
    - OLD snapshot details (location, tags),
    - LATEST snapshot details (location, tags),
    - `distance_in_meters` and `hookline`,
    - whether the **OLD** APT was inside a BFP, and whether the **NEW** APT is inside a BFP.

In [ ]:
%sql
-- Create the databases that hold the OLD and LATEST snapshot Delta tables for scoping.
-- Run this once before the snapshot loader cells below.
CREATE DATABASE IF NOT EXISTS rooftop_regression_scoping_db_old;
CREATE DATABASE IF NOT EXISTS rooftop_regression_scoping_db_new;

In [ ]:
%scala
// Step 1 - Load the OLD snapshot into a Delta table (filtered to country = ESP).
// Pin OLD_REVISION to the older revision you want to compare the latest snapshot against.

import com.databricks.dbutils_v1.DBUtilsHolder
import com.fasterxml.jackson.databind.{DeserializationFeature, ObjectMapper}
import com.tomtom.addressing.bulk.commons.model.LayerVersions
import com.tomtom.orbis.addressing.bulk.commons.spark.SparkHelper
import com.tomtom.addressing.bulk.commons.config.ConfigLoader
import com.tomtom.addressing.bulk.scala.load.LoadFreshSnapshotData
import com.tomtom.addressing.bulk.commons.config.SourceConfigLoader
import com.tomtom.orbis.addressing.bulk.commons.repository.OrbisElementRepository
import org.apache.spark.sql.functions._

val LAYER_ID = 14533
val OLD_DATABASE = "rooftop_regression_scoping_db_old"
val COUNTRY_ISO = "ESP"

// >>> Set the OLD snapshot revision id here <<<
val OLD_REVISION = 42372254L

val mapper = new ObjectMapper()
  .configure(DeserializationFeature.FAIL_ON_UNKNOWN_PROPERTIES, false)
  .configure(DeserializationFeature.ACCEPT_EMPTY_STRING_AS_NULL_OBJECT, true)
  .configure(DeserializationFeature.ACCEPT_SINGLE_VALUE_AS_ARRAY, true)

val env = "PROD"
implicit val sparky = spark
ConfigLoader.forEnvironment(env.toLowerCase)

println(s"Loading OLD snapshot revision: $OLD_REVISION")
val versionsBuilder = LayerVersions.builder()
versionsBuilder.layer(LAYER_ID, OLD_REVISION)
val versionMetadata: String = mapper.writeValueAsString(versionsBuilder.build())
DBUtilsHolder.dbutils.widgets.text("layer-versions", versionMetadata)

SparkHelper.init(OLD_DATABASE)
new LoadFreshSnapshotData().run()

// Persist the loaded old snapshot, keeping only APTs whose metadata:country tag = ESP
val oldSnapshotTable = s"${OLD_DATABASE}.apt_snapshot_revision_${OLD_REVISION}_${LAYER_ID}"
val oldAptDS = new OrbisElementRepository(LAYER_ID.toString).readAll
  .filter(exists(col("tags"), t =>
    t.getField("tagKey").getField("key") === "metadata:country"
    && t.getField("value") === COUNTRY_ISO))
oldAptDS.write.format("delta").mode("overwrite").saveAsTable(oldSnapshotTable)
println(s"OLD snapshot written to: $oldSnapshotTable")
display(oldAptDS)

In [ ]:
%scala
// Step 2 - Load the LATEST snapshot into a Delta table (filtered to country = ESP).
// The newest revision is resolved automatically via LoadSnapshotInfo.getLatestRevisionId.

import com.databricks.dbutils_v1.DBUtilsHolder
import com.fasterxml.jackson.databind.{DeserializationFeature, ObjectMapper}
import com.tomtom.addressing.bulk.commons.model.LayerVersions
import com.tomtom.orbis.addressing.bulk.commons.spark.SparkHelper
import com.tomtom.addressing.bulk.commons.config.ConfigLoader
import com.tomtom.addressing.bulk.scala.load.LoadFreshSnapshotData
import com.tomtom.orbis.addressing.bulk.commons.odp.LoadSnapshotInfo
import com.tomtom.addressing.bulk.commons.config.SourceConfigLoader
import com.tomtom.orbis.addressing.bulk.commons.repository.OrbisElementRepository
import org.apache.spark.sql.functions._

val LAYER_ID = 14533
val NEW_DATABASE = "rooftop_regression_scoping_db_new"
val COUNTRY_ISO = "ESP"

val mapper = new ObjectMapper()
  .configure(DeserializationFeature.FAIL_ON_UNKNOWN_PROPERTIES, false)
  .configure(DeserializationFeature.ACCEPT_EMPTY_STRING_AS_NULL_OBJECT, true)
  .configure(DeserializationFeature.ACCEPT_SINGLE_VALUE_AS_ARRAY, true)

val env = "PROD"
implicit val sparky = spark
ConfigLoader.forEnvironment(env.toLowerCase)

val latestRevision = LoadSnapshotInfo.getLatestRevisionId(LAYER_ID)
println(s"Loading LATEST snapshot revision: ${latestRevision.get}")
val versionsBuilder = LayerVersions.builder()
versionsBuilder.layer(LAYER_ID, latestRevision.get)
val versionMetadata: String = mapper.writeValueAsString(versionsBuilder.build())
DBUtilsHolder.dbutils.widgets.text("layer-versions", versionMetadata)

SparkHelper.init(NEW_DATABASE)
new LoadFreshSnapshotData().run()

// Persist the loaded latest snapshot, keeping only APTs whose metadata:country tag = ESP
val latestSnapshotTable = s"${NEW_DATABASE}.apt_snapshot_revision_${latestRevision.get}_${LAYER_ID}"
val latestAptDS = new OrbisElementRepository(LAYER_ID.toString).readAll
  .filter(exists(col("tags"), t =>
    t.getField("tagKey").getField("key") === "metadata:country"
    && t.getField("value") === COUNTRY_ISO))
latestAptDS.write.format("delta").mode("overwrite").saveAsTable(latestSnapshotTable)
println(s"LATEST snapshot written to: $latestSnapshotTable")
display(latestAptDS)

In [ ]:
%scala
// Step 3 - Read the OLD snapshot Delta table as a Dataset[OrbisElement].
// Enrich each row with country, the unsigned colon-separated orbisIdString, and metadata:updated (epoch).

import org.apache.spark.sql.{Dataset, Encoders}
import org.apache.spark.sql.functions._
import com.tomtom.orbis.io.spark.model.{Id, OrbisElement}
import java.lang.{Long => JLong}

// Build the colon-separated unsigned id string: {layerId}:{unsignedHigh}:{unsignedLow}
def convertOrbisIdToString(orbisId: Id): String = {
  val COLON_SEPARATOR = ":"
  Seq(orbisId.layerId.getOrElse(14533).toString,
      JLong.toUnsignedString(orbisId.high),
      JLong.toUnsignedString(orbisId.low)).mkString(COLON_SEPARATOR)
}
val convertOrbisIdUDF = udf((orbisId: Id) => convertOrbisIdToString(orbisId))

val countryTagKey = "metadata:country"
val updatedTagKey = "metadata:updated"

// >>> Point this at the OLD snapshot table written in Step 1 <<<
val oldSnapshotTable = "rooftop_regression_scoping_db_old.apt_snapshot_revision_42372254_14533"

val oldSnapshotDS: Dataset[OrbisElement] =
  spark.table(oldSnapshotTable).as[OrbisElement](Encoders.product[OrbisElement])

val oldSnapshotEnriched = oldSnapshotDS
  .withColumn("country",
    filter(col("tags"), t => t.getField("tagKey").getField("key") === countryTagKey)
      .getItem(0).getField("value"))
  .withColumn("orbisIdString", convertOrbisIdUDF(col("id")))
  // metadata:updated value is an epoch timestamp string -> cast to long for the date filter in Step 6
  .withColumn("updated_epoch",
    filter(col("tags"), t => t.getField("tagKey").getField("key") === updatedTagKey)
      .getItem(0).getField("value").cast("long"))

println(s"OLD snapshot read from: $oldSnapshotTable, count: ${oldSnapshotEnriched.count()}")
display(oldSnapshotEnriched)

In [ ]:
%scala
// Step 4 - Read the LATEST snapshot Delta table as a Dataset[OrbisElement].
// Enrich each row with country and the unsigned colon-separated orbisIdString (UDF defined in Step 3).

import org.apache.spark.sql.{Dataset, Encoders}
import org.apache.spark.sql.functions._
import com.tomtom.orbis.io.spark.model.OrbisElement

// >>> Point this at the LATEST snapshot table written in Step 2 <<<
val latestSnapshotTable = "rooftop_regression_scoping_db_new.apt_snapshot_revision_42621089_14533"

val latestSnapshotDS: Dataset[OrbisElement] =
  spark.table(latestSnapshotTable).as[OrbisElement](Encoders.product[OrbisElement])

val latestSnapshotEnriched = latestSnapshotDS
  .withColumn("country",
    filter(col("tags"), t => t.getField("tagKey").getField("key") === countryTagKey)
      .getItem(0).getField("value"))
  .withColumn("orbisIdString", convertOrbisIdUDF(col("id")))

println(s"LATEST snapshot read from: $latestSnapshotTable, count: ${latestSnapshotEnriched.count()}")
display(latestSnapshotEnriched)

In [ ]:
%scala
// Step 5 - Filter both snapshots to country ISO = ESP.
// (Snapshots are already loaded ESP-only in Steps 1-2; this re-applies the filter defensively.)

import org.apache.spark.sql.functions._

val COUNTRY_ISO = "ESP"

val oldEsp = oldSnapshotEnriched.filter(col("country") === COUNTRY_ISO)
val latestEsp = latestSnapshotEnriched.filter(col("country") === COUNTRY_ISO)

println(s"OLD ESP count: ${oldEsp.count()}")
println(s"LATEST ESP count: ${latestEsp.count()}")

In [ ]:
%scala
// Step 6 - Filter the OLD snapshot to APTs updated after 1st Feb 2026 (the regression window start).
// metadata:updated is an epoch timestamp; adjust the constant if your values are in seconds, not millis.

import org.apache.spark.sql.functions._

// 2026-02-01T00:00:00Z in epoch MILLISECONDS. Use 1769904000L if metadata:updated is in seconds.
val FEB_FIRST_EPOCH_MILLIS = 1769904000000L

val oldEspUpdated = oldEsp.filter(col("updated_epoch") > FEB_FIRST_EPOCH_MILLIS)

println(s"OLD ESP updated after 1st Feb count: ${oldEspUpdated.count()}")
display(oldEspUpdated)

In [ ]:
%scala
// Steps 7-8 - Left join OLD (updated, ESP) with LATEST (ESP) on orbisIdString, then add geometry columns:
// old_wkt / latest_wkt (POINT(lng lat)), distance_in_meters (Haversine) and hookline (LINESTRING old->latest).

import org.apache.spark.sql.functions._

val oldRenamed = oldEspUpdated
  .select(col("orbisIdString"), col("lat"), col("lng"), col("tags"))
  .withColumn("old_wkt", concat(lit("POINT("), col("lng"), lit(" "), col("lat"), lit(")")))
  .withColumnRenamed("lat", "old_lat")
  .withColumnRenamed("lng", "old_lon")
  .withColumnRenamed("tags", "old_tags")

val latestRenamed = latestEsp
  .select(col("orbisIdString"), col("lat"), col("lng"), col("tags"))
  .withColumn("latest_wkt", concat(lit("POINT("), col("lng"), lit(" "), col("lat"), lit(")")))
  .withColumnRenamed("lat", "latest_lat")
  .withColumnRenamed("lng", "latest_lon")
  .withColumnRenamed("tags", "latest_tags")

// Haversine distance in meters (mean earth radius) - avoids Sedona for the distance calc
val EARTH_RADIUS_M = 6371008.8
val dLat = radians(col("latest_lat") - col("old_lat"))
val dLon = radians(col("latest_lon") - col("old_lon"))
val haversine =
  pow(sin(dLat / 2), 2) +
  cos(radians(col("old_lat"))) * cos(radians(col("latest_lat"))) * pow(sin(dLon / 2), 2)
val distanceMeters = lit(2 * EARTH_RADIUS_M) * asin(sqrt(haversine))

val oldVsLatest = oldRenamed
  .join(latestRenamed, Seq("orbisIdString"), "left")
  .withColumn("distance_in_meters",
    when(col("old_wkt").isNotNull && col("latest_wkt").isNotNull, distanceMeters))
  .withColumn("hookline",
    when(col("old_wkt").isNotNull && col("latest_wkt").isNotNull,
      concat(lit("LINESTRING("),
        col("old_lon"), lit(" "), col("old_lat"), lit(", "),
        col("latest_lon"), lit(" "), col("latest_lat"), lit(")"))))
  .select(
    col("orbisIdString"),
    col("old_lat"), col("old_lon"), col("old_wkt"), col("old_tags"),
    col("latest_lat"), col("latest_lon"), col("latest_wkt"), col("latest_tags"),
    col("distance_in_meters"),
    col("hookline"))

// Persist so the Scala/Sedona BFP cells below can read it back
oldVsLatest.write.format("delta").mode("overwrite")
  .saveAsTable("rooftop_regression_scoping_db_new.old_vs_latest_esp")
println(s"old-vs-latest rows: ${oldVsLatest.count()}")
display(oldVsLatest)

In [ ]:
%scala
// Step 9 (a) - Refresh the MCR Building Footprint delta table (preprocess_prod.bfp) via LoadDataFromMcr.
// Loads the fresh snapshot + MCR BFP data so the ESP BFP read in the next cell has data to query.

import org.apache.spark.sql.{Dataset, SparkSession}
import com.tomtom.orbis.addressing.bulk.commons.spark.SparkHelper
import com.tomtom.addressing.bulk.commons.config.ConfigLoader
import com.tomtom.addressing.bulk.scala.load.{LoadFreshSnapshotData, LoadDataFromMcr}
import com.databricks.dbutils_v1.DBUtilsHolder
import com.tomtom.addressing.bulk.commons.config.SourceConfigLoader

// Default the environment widget so the getter below never fails (matches env = PROD used above)
// dbutils.widgets.text("environment", "PROD")
val env = "PROD" //dbutils.widgets.get("environment").toUpperCase()
println(s"Environment Name $env")

val config = SourceConfigLoader.getConfig

val deltaConfig = config.getTableMapping.getOrDefault(env, "preprocess_dev.source_data")
val Array(databaseName, tableName) = deltaConfig.split("\\.")
val databaseName = NEW_DATABASE
println(s"Database Name: $databaseName")
println(s"Table Name: $tableName")

implicit val sparky = spark
implicit val envConfig = ConfigLoader.forEnvironment(env.toLowerCase)
SparkHelper.init(databaseName)

new LoadDataFromMcr().run()

In [ ]:
%scala
// Step 9 (b) - Read the ESP Building Footprint (BFP) polygons from preprocess_prod.bfp.
// The MCR BFP table was (re)populated in the previous cell; filter to the ESP license zone.

import org.apache.spark.sql.functions._
import org.apache.sedona.spark.SedonaContext

// Sedona is required for the ST_Intersects spatial join in the next cells
val sedona = SedonaContext.create(spark)

val espBfpDataset = spark.table("preprocess_prod.bfp")
  .filter(col("licenseZone") === "ESP")
  .filter(col("wkt").isNotNull)
  .withColumn("bfp_geom", expr("ST_GeomFromWKT(wkt)"))

println(s"ESP BFP polygon count: ${espBfpDataset.count()}")
display(espBfpDataset)

In [ ]:
%scala
// Step 10 - Spatially check whether each APT's OLD location falls inside a building footprint.
// Left ST_Intersects join old_wkt against the BFP polygons, collapsed to one old_inside_bfp flag per APT.

import org.apache.spark.sql.functions._

val oldVsLatestTbl = spark.table("rooftop_regression_scoping_db_new.old_vs_latest_esp")

val bfpGeoms = espBfpDataset.select("bfp_geom").repartition(250, expr("ST_GeoHash(bfp_geom, 5)"))

val oldInBfp = oldVsLatestTbl.select("orbisIdString", "old_wkt").repartition(1000)
  .withColumn("old_geom", expr("ST_GeomFromText(old_wkt)")).alias("apt")
  .join(bfpGeoms.alias("bfp"), expr("ST_Intersects(bfp.bfp_geom, apt.old_geom)"), "left")
  .groupBy("orbisIdString")
  .agg(max(when(col("bfp.bfp_geom").isNotNull, true).otherwise(false)).as("old_inside_bfp"))

println(s"APTs evaluated for OLD-in-BFP: ${oldInBfp.count()}")
display(oldInBfp)

In [ ]:
%scala
// Step 11 - Same ST_Intersects check for the LATEST (new) location, then assemble the final scoped dataset.
// Final = old details + latest details + distance/hookline + old_inside_bfp + new_inside_bfp flags.

import org.apache.spark.sql.functions._

val newInBfp = oldVsLatestTbl.select("orbisIdString", "latest_wkt")
  .filter(col("latest_wkt").isNotNull).repartition(1000)
  .withColumn("latest_geom", expr("ST_GeomFromText(latest_wkt)")).alias("apt")
  .join(bfpGeoms.alias("bfp"), expr("ST_Intersects(bfp.bfp_geom, apt.latest_geom)"), "left")
  .groupBy("orbisIdString")
  .agg(max(when(col("bfp.bfp_geom").isNotNull, true).otherwise(false)).as("new_inside_bfp"))

val finalScoped = oldVsLatestTbl
  .join(oldInBfp, Seq("orbisIdString"), "left")
  .join(newInBfp, Seq("orbisIdString"), "left")
  .withColumn("new_inside_bfp", coalesce(col("new_inside_bfp"), lit(false)))

// Persist the final scoped relocated-APT set for validation / Data Correctness trigger
finalScoped.write.format("delta").mode("overwrite")
  .saveAsTable("rooftop_regression_scoping_db_new.relocated_apt_scoped_esp")

println(s"Final scoped APT count: ${finalScoped.count()}")
display(finalScoped)